In [ ]:
# first run `uv pip install google-genai openai python-dotenv`
import os
import time
from google import genai
from google.genai.errors import APIError
from openai import OpenAI
from dotenv import load_dotenv

# Load your secret keys from the .env file
load_dotenv()

# ==============================================================================
# OPENAI (CHATGPT) MODEL CODES FOR API
# Note: The OpenAI API is technically Pay-As-You-Go (you buy credits), 
# but these are the models powering the free/standard tiers of the ChatGPT app:
#
# 1. "gpt-4o-mini"   -> The current default fast, cheap, and smart model.
#                       (Best equivalent to Gemini Flash. Highly recommended).
#
# 2. "gpt-4o"        -> The flagship, most intelligent model. Slower and costs more credits.
# 
# 3. "gpt-3.5-turbo" -> The older legacy model. (Now mostly replaced by gpt-4o-mini).
# ==============================================================================

# Initialize both AI clients securely using the environment variables
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
def llm(prompt):
    """
    Main function to ask the AI a question. 
    It prioritizes Gemini, but falls back to ChatGPT if Gemini is overloaded.
    """
    
    # ---------------------------------------------------------
    # ROUTE 1: Try Gemini First
    # ---------------------------------------------------------
    try:
        print("Attempting to use Gemini...")
        
        # We call the new google-genai SDK method here
        response = gemini_client.models.generate_content(
            model='gemini-3.1-flash-lite', 
            contents=prompt
        )
        return response.text

    # ---------------------------------------------------------
    # ROUTE 2: Catch Gemini Server Errors and Fallback to ChatGPT
    # ---------------------------------------------------------
    except APIError as e:
        # Check if the error code means the server is just busy or rate-limited
        # 503 = Service Unavailable (Busy)
        # 429 = Too Many Requests (Rate limit hit)
        # 500 = Internal Server Error (Google is having a glitch)
        if e.code in [503, 429, 500]:
            print(f"Gemini unavailable (Error {e.code}). Falling back to ChatGPT...")
            # Route the exact same prompt to our backup function
            return ask_chatgpt(prompt)
        else:
            # If it's a different error (like an invalid API key), 
            # we want the program to crash so we can fix the bug.
            raise e
            
    except Exception as e:
        # Catch-all for basic internet connection drops or timeouts
        print(f"Gemini failed with a general error: {e}. Falling back to ChatGPT...")
        return ask_chatgpt(prompt)


def ask_chatgpt(prompt):
    """
    Helper function to handle the OpenAI (ChatGPT) API call.
    This only runs if Gemini fails.
    """
    try:
        # We use the standard OpenAI chat completions method here
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini", # <-- You can swap this with the models listed at the top!
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        # Extract the text from the OpenAI response structure
        return response.choices[0].message.content
        
    except Exception as e:
        # If BOTH Google and OpenAI servers are down, return a hard failure message
        return f"System Failure: Both Gemini and ChatGPT are unavailable. ChatGPT error: {e}"


# ==============================================================================
# TEST THE PIPELINE
# ==============================================================================
result = llm("Explain quantum computing in simple terms")

print("\n--- Final Output ---")
print(result)

Attempting to use Gemini...

--- Final Output ---
To understand quantum computing, first think about how a regular computer works.

### The Regular Computer (The Library)
A regular computer (like your phone or laptop) uses **bits**. Think of a bit like a light switch: it is either **ON (1)** or **OFF (0)**. Everything you do on a computer—watching videos, typing emails, or playing games—is just a massive collection of billions of these little light switches flipping on and off.

### The Quantum Computer (The Magic Coin)
A quantum computer uses **qubits**. 

Imagine a coin. A regular computer bit is like a coin lying on a table. It is either **Heads** or **Tails**. 

A **qubit** is like a coin **spinning on the table**. While it’s spinning, is it Heads or Tails? It’s kind of **both at the same time.** In the world of quantum physics, this is called **Superposition.**

### Why does this matter?
Because qubits can be both 0 and 1 at the same time, they can hold much more information than 

# 1.2 Embeddings

```bash
uv add sentence-transformers
```

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [ ]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
v1.dot(dv)

np.float32(0.32332397)

In [ ]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [ ]:
v2.dot(dv)

np.float32(0.019730438)

## 1.3 Embedding Our Dataset


In [ ]:
import os
os.getcwd()

'/workspaces/LLM-Zoomcamp/02-vector-search'

In [ ]:
import os

os.listdir()

['0.vector.ipynb',
 '.gitkeep',
 '__pycache__',
 'faq_vectors2.db',
 'rag_helper.py',
 'ingest.py']

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [ ]:
import numpy as np
X = np.array(vectors)

In [ ]:
X.shape

(1350, 384)

# 1.4 Vector Search

In [ ]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)
print(v_query)

[ 2.13904120e-02 -7.39799663e-02  1.42068556e-03  2.13816613e-02
  2.45113149e-02  3.15582752e-02 -1.10839702e-01 -1.05017498e-01
 -6.18258938e-02 -6.42311852e-03  3.72399064e-03  9.06393602e-02
 -9.49941017e-03  6.53976873e-02  1.10946558e-02 -2.10097339e-02
 -3.35125513e-02 -4.31677327e-02  9.96348727e-03  1.41969863e-02
 -6.40414879e-02 -7.04182126e-03 -7.91188031e-02  5.80030754e-02
  1.30212074e-03  4.19733534e-03  5.70979156e-02  6.39447570e-02
  2.49902904e-02 -3.95876691e-02 -3.79506797e-02  2.70394552e-02
  1.79423206e-02  1.72272157e-02  3.43311131e-02  9.29055270e-03
  5.86054735e-02 -4.97789569e-02 -5.05366083e-03  4.34328541e-02
 -1.56622957e-02 -2.97534503e-02 -5.13325352e-03  5.13414629e-02
  6.16060290e-03  6.86980635e-02 -1.29505778e-02 -5.61938696e-02
 -1.08265011e-02  5.96683845e-02  5.29939681e-02 -3.42755206e-02
 -4.15274203e-02 -5.43295704e-02 -7.14875804e-03  1.28990725e-01
  2.09546369e-02  1.25907324e-02 -1.04924910e-01  1.57250781e-02
 -7.83412978e-02 -2.66572

In [ ]:
scores = X.dot(v_query)
print(scores)

[ 0.48740575  0.20991933  0.762941   ... -0.08637968  0.03759793
 -0.03037044]


In [ ]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [ ]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [ ]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [ ]:
top5 = np.argsort(scores)[-5:]

In [ ]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [ ]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'

## 1.5 Vector Search with minsearch


In [ ]:
print(len(X))
print(len(documents))

1350
1350


In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [ ]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

## 1.6 RAG with Vector Search


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=gemini_client,
)

In [ ]:
query = "I just found out about the program, can I still sign up?"
answer = vector_assistant.rag(query)
print(answer)

NameError: name 'vector_assistant' is not defined

In [ ]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [ ]:
query = "the program has already begun, can I still sign up?"
answer = vector_assistant.rag(query)
print(answer)

Yes, you can still join. If you want to receive a certificate, you just need to ensure you submit your project while submissions are still being accepted.


## 1.7 Vector Search with sqlitesearch


In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)


In [ ]:
vs_index.fit(vectors, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [ ]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [ ]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [ ]:
vs_index.close()

In [ ]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [ ]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )
        
    # ADD THIS: Overrides the default RAGBase method to use Gemini correctly
    def llm(self, prompt):
        full_prompt = f"{self.instructions}\n\n{prompt}"
        
        try:
            # This handles the new Gemini SDK (from google import genai)
            response = self.llm_client.models.generate_content(
                model='gemini-3.1-flash-lite', 
                contents=full_prompt
            )
            return response.text
            
        except AttributeError:
            # Fallback for the legacy SDK (import google.generativeai as genai)
            from google import genai
            model_instance = gemini_client.models.generate_content(
            model='gemini-3.1-flash-lite', 
            contents=prompt)
            response = model_instance.generate_content(full_prompt)
            return response.text


# Initialise the assistant again
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=gemini_client,
)

In [ ]:
print(type(gemini_client))

<class 'google.genai.client.Client'>


In [ ]:
query = "I just found out about the programme, can I still sign up?"
answer = vector_assistant.rag(query)
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [ ]:
query = "I just found out about the programme, can I still sign up?"
answer = vector_assistant.rag(query)
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [ ]:
vs_index.close()

## 1.8 Vector Search with PGVector

```bash
docker run -it \
    --name pgvector \
    -e POSTGRES_USER=user \
    -e POSTGRES_PASSWORD=pswd \
    -e POSTGRES_DB=faq \
    -v pgvector_data:/var/lib/postgresql/data \
    -p 5432:5432 \
    pgvector/pgvector:pg17

uv add psycopg[binary]

```

In [ ]:
from tqdm.auto import tqdm

from ingest import load_faq_data
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = load_faq_data()
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

In [ ]:
from tqdm.auto import tqdm

from ingest import load_faq_data
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = load_faq_data()
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]